In [1]:
import numpy as np
import pandas as pd
import os
import wandb
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize
import torch
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
import gc

In [ ]:
os.environ["WANDB_API_KEY"] = "API Key"
wandb.login()

wandb: Currently logged in as: nk23041720 (nk23041720-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
wandb_config = {
    "model_name": "microsoft/mdeberta-v3-base",
    "task": "nepali_multiclass_classification",
    "learning_rate": 2e-5,  
    "batch_size": 8,
    "gradient_accumulation_steps": 2,  
    "num_epochs": 10,  
    "max_length": 256,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-8, 
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "max_grad_norm": 1.0,  
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1,
    "random_seed": 42,
    "warmup_ratio": 0.1,  
    "eval_steps": 1000,  
    "logging_steps": 100,
    "save_steps": 1000,
    "early_stopping_patience": 15,
    "label_smoothing": 0.1, 
    "fp16": False,  
     "bf16": False,
}

In [4]:
run = wandb.init(
    project="topic-modeling-25k-experiments",
    name="mdBERTa-25k-2e5-16-256-10",
    config=wandb_config,
    tags=["nepali", "bert", "classification", "mdberta"]
)

wandb: creating run


wandb: Tracking run with wandb version 0.21.3


wandb: Run data is saved locally in /home/rupak/Desktop/Topic-Modeling /topic modeling 25k/code/wandb/run-20251017_104641-yeiitkhc
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run mdBERTa-25k-2e5-16-256-10


wandb: ⭐️ View project at https://wandb.ai/nk23041720-kathmandu-university/topic-modeling-25k-experiments


wandb: 🚀 View run at https://wandb.ai/nk23041720-kathmandu-university/topic-modeling-25k-experiments/runs/yeiitkhc


In [5]:
df = pd.read_excel("/home/rupak/Desktop/Topic-Modeling /dataset/topic-modeling-25k-dataset.xlsx")
df.head()

,sentenceid,relevant_sentences,domainid
0,COM_D3E_S1_S2_6369,अक्षय कोषको आकार एक करोड रूपिया पुर्‍याउने तत्...,D3E
1,COM_D3E_S1_S2_4941,जेहेन्दार विद्यार्थीलाई छात्रवृत्ति प्रदान गरि...,D3E
2,COM_D3E_S1_S2_1399,अक्षराङ्कन पद्धतिमा शुरूमा आठ ग्रेडको व्यवस्था...,D3E
3,COM_D3E_S1_S2_1390,अक्षराङ्कन लागू हुनु पहिला विद्यार्थी दुई कित्...,D3E
4,COM_D3E_S1_S2_1049,अक्सिटोसिनले अध्यात्मिक सोच बढाउनमा सहायक पाइए...,D3E


In [6]:
df = df.sample(frac=1, random_state=wandb.config.random_seed).reset_index(drop=True)
df.head()

,sentenceid,relevant_sentences,domainid
0,COM_D4C_S1_S2_4940,सन्तानेश्वरमा पनि हेलिकप्टरबाट आउन चाहने तीर्थ...,D4C
1,COM_D4C_S1_S2_5909,विभागका अनुसार शरद ऋतुका लागि उत्तम मानिने मना...,D4C
2,COM_D2H_S1_S2_2968,चीनमा अहिले सङ्क्रमण हुन सक्ने जनावर मार्न र ख...,D2H
3,D1A_S0_B1_231,यो सिजनको मकै खेती तराईमा गरिन्छ।,D1A
4,D1A_S0_B1_4467,निजी क्षेत्र र सहकारी क्षेत्रले सरकारको उक्त न...,D1A


In [7]:
unique_labels = sorted(df["domainid"].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
df["labels"] = df["domainid"].map(label2id)

In [8]:
dataset = Dataset.from_pandas(df[['relevant_sentences', 'labels']])
dataset = dataset.rename_column('relevant_sentences', 'text')

In [9]:
dataset.shape

(25006, 2)

In [10]:
train_test_split = dataset.train_test_split(
    test_size=0.2,
    seed=wandb.config.random_seed,
    shuffle=True
)
train_dataset = train_test_split['train']

In [11]:
val_test_split = train_test_split['test'].train_test_split(
    test_size=0.5,
    seed=wandb.config.random_seed,
    shuffle=True
)
val_dataset = val_test_split['train']  # 10% of total
test_dataset = val_test_split['test']   # 10% of total

In [12]:
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [13]:
model_name = wandb.config.model_name
tokenizer = AutoTokenizer.from_pretrained(model_name)

/home/rupak/Desktop/Topic-Modeling /venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [14]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=False,  # Dynamic padding with data collator
        truncation=False,
        max_length=wandb.config.max_length
    )

In [15]:
dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])

Map:   0%|          | 0/20004 [00:00<?, ? examples/s]

Map:   0%|          | 0/2501 [00:00<?, ? examples/s]

Map:   0%|          | 0/2501 [00:00<?, ? examples/s]

In [16]:
num_labels = len(unique_labels)
num_labels

5

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.torch.float32,
    device_map="auto",
    trust_remote_code=False,
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

In [19]:
total_steps = (len(dataset["train"]) // (wandb.config.batch_size * wandb.config.gradient_accumulation_steps)) * wandb.config.num_epochs
warmup_steps = int(total_steps * wandb.config.warmup_ratio)

In [20]:
training_args = TrainingArguments(
    output_dir="/home/rupak/Desktop/Topic-Modeling /topic modeling 25k/checkpoint/mdberta-25k-2e5-256-16-10",
    eval_strategy="steps",
    eval_steps=wandb.config.eval_steps,
    save_strategy="steps",
    save_steps=wandb.config.save_steps,
    learning_rate=wandb.config.learning_rate,
    per_device_train_batch_size=wandb.config.batch_size,
    per_device_eval_batch_size=wandb.config.batch_size,
    gradient_accumulation_steps=wandb.config.gradient_accumulation_steps,
    num_train_epochs=wandb.config.num_epochs,
    weight_decay=wandb.config.weight_decay,
    adam_beta1=wandb.config.adam_beta1,
    adam_beta2=wandb.config.adam_beta2,
    adam_epsilon=wandb.config.adam_epsilon,
    max_grad_norm=wandb.config.max_grad_norm,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_weighted",
    greater_is_better=True,
    save_total_limit=3,
    label_smoothing_factor=wandb.config.label_smoothing,
    # Enhanced logging
    logging_dir='./logs',
    logging_steps=wandb.config.logging_steps,
    logging_first_step=True,
    # Wandb integration
    report_to="wandb",
    run_name=f"nepberta-optimized-{wandb.run.id}",
    # Full precision training (FP32)
    fp16=wandb.config.fp16,
    bf16=wandb.config.bf16,
    dataloader_drop_last=False,
    eval_accumulation_steps=None,
    # Additional optimization
    dataloader_num_workers=2,
    remove_unused_columns=False,
    push_to_hub=False,
)

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Get probabilities for AUROC
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    # Calculate comprehensive metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )

    # Macro averages for unbiased evaluation
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    # Calculate AUROC (for multiclass)
    if num_labels > 2:
        labels_binarized = label_binarize(labels, classes=range(num_labels))
        try:
            auroc_weighted = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='weighted')
            auroc_macro = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='macro')
        except ValueError:
            auroc_weighted = 0.0
            auroc_macro = 0.0
    else:
        auroc_weighted = roc_auc_score(labels, probabilities[:, 1])
        auroc_macro = auroc_weighted

    metrics = {
        "accuracy": accuracy,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "auroc_weighted": auroc_weighted,
        "auroc_macro": auroc_macro
    }

    return metrics

In [22]:
class CustomTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_accuracy_history = []
        self._last_train_metrics = {}



    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss and optionally calculate training accuracy
        """
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss = outputs.get("loss")

        # Calculate training accuracy periodically
        if (self.state.global_step % (self.args.logging_steps * 2) == 0 and
            labels is not None and self.state.global_step > 0):
            with torch.no_grad():
                predictions = torch.argmax(outputs.logits, dim=-1)
                accuracy = (predictions == labels).float().mean().item()
                self._last_train_metrics = {"train_accuracy": accuracy}

        return (loss, outputs) if return_outputs else loss

In [23]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=wandb.config.early_stopping_patience)]
)

/tmp/ipykernel_756685/3450341322.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [24]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision Macro,Recall Macro,F1 Macro,Auroc Weighted,Auroc Macro
1000,0.505700,0.523028,0.841663,0.850197,0.841663,0.842300,0.850136,0.843986,0.843341,0.965430,0.965941
2000,0.384200,0.472351,0.861655,0.869263,0.861655,0.860752,0.868204,0.864736,0.861774,0.974158,0.974534
3000,0.304200,0.466291,0.869252,0.869991,0.869252,0.868198,0.869884,0.871451,0.869262,0.973582,0.974042
4000,0.194600,0.523235,0.881248,0.882731,0.881248,0.880984,0.882860,0.882893,0.881871,0.975685,0.976107
5000,0.288200,0.468508,0.880448,0.881562,0.880448,0.879298,0.881597,0.882579,0.880445,0.978081,0.978455
6000,0.175600,0.595979,0.881248,0.882204,0.881248,0.879976,0.882147,0.883531,0.881148,0.974808,0.975288
7000,0.134000,0.663512,0.882847,0.884105,0.882847,0.882002,0.884090,0.885015,0.883098,0.973248,0.973705
8000,0.155400,0.707383,0.880048,0.880393,0.880048,0.879388,0.880772,0.881779,0.880460,0.973560,0.973970
9000,0.105300,0.764610,0.876449,0.877430,0.876449,0.874697,0.877270,0.878941,0.875936,0.969305,0.969802
10000,0.067500,0.745228,0.887245,0.887213,0.887245,0.886883,0.887516,0.888736,0.887783,0.971620,0.972013


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


TrainOutput(global_step=12510, training_loss=0.2446709101339229, metrics={'train_runtime': 2398.8001, 'train_samples_per_second': 83.392, 'train_steps_per_second': 5.215, 'total_flos': 5840188686810192.0, 'train_loss': 0.2446709101339229, 'epoch': 10.0})

In [25]:
test_results = trainer.evaluate(eval_dataset=dataset["test"])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
for key, value in test_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').replace('_', ' ').title()
        print(f"{metric_name:<20}: {value:.4f}")

Loss                : 0.6577
Accuracy            : 0.8984
Precision Weighted  : 0.8982
Recall Weighted     : 0.8984
F1 Weighted         : 0.8982
Precision Macro     : 0.8969
Recall Macro        : 0.8969
F1 Macro            : 0.8968
Auroc Weighted      : 0.9766
Auroc Macro         : 0.9764
Runtime             : 6.3330
Samples Per Second  : 394.9130
Steps Per Second    : 49.4230


In [27]:
# Get detailed predictions for test set
test_predictions = trainer.predict(dataset["test"])
test_logits = test_predictions.predictions
test_labels = test_predictions.label_ids
test_pred_labels = np.argmax(test_logits, axis=-1)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
# Convert back to original domain labels for interpretability
test_pred_domains = [id2label[pred] for pred in test_pred_labels]
test_true_domains = [id2label[label] for label in test_labels]

In [29]:
report = classification_report(
    test_labels, 
    test_pred_labels, 
    target_names=[id2label[i] for i in range(num_labels)],
    digits=4
)

In [30]:
print(report)

              precision    recall  f1-score   support

         D1A     0.9133    0.9287    0.9209       533
         D2H     0.8684    0.8827    0.8755       486
         D3E     0.9162    0.8918    0.9038       527
         D4C     0.9368    0.9556    0.9461       496
         D5G     0.8498    0.8257    0.8376       459

    accuracy                         0.8984      2501
   macro avg     0.8969    0.8969    0.8968      2501
weighted avg     0.8982    0.8984    0.8982      2501



In [31]:
wandb.finish()

wandb: updating run metadata


wandb: uploading wandb-summary.json; uploading config.yaml


wandb:                                                                                


wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▃▄▆▆▆▆▆▅▇▆▆█
wandb:        eval/auroc_macro ▁▆▆▇█▆▅▅▃▄▄▄▇
wandb:     eval/auroc_weighted ▁▆▆▇█▆▅▅▃▄▄▄▇
wandb:           eval/f1_macro ▁▃▄▆▆▆▆▆▅▇▆▇█
wandb:        eval/f1_weighted ▁▃▄▆▆▆▆▆▅▇▆▆█
wandb:               eval/loss ▂▁▁▂▁▄▅▆▇▇██▅
wandb:    eval/precision_macro ▁▄▄▆▆▆▆▆▅▇▆▆█
wandb: eval/precision_weighted ▁▄▄▆▆▆▆▅▅▆▆▆█
wandb:       eval/recall_macro ▁▄▅▆▆▆▆▆▆▇▇▇█
wandb:    eval/recall_weighted ▁▃▄▆▆▆▆▆▅▇▆▆█
wandb:                     +21 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.89844
wandb:        eval/auroc_macro 0.97639
wandb:     eval/auroc_weighted 0.97661
wandb:           eval/f1_macro 0.89679
wandb:        eval/f1_weighted 0.8982
wandb:               eval/loss 0.65771
wandb:    eval/precision_macro 0.89688
wandb: eval/precision_weighted 0.89818
wandb:       eval/recall_macro 0.89692
wandb:    eval/recall_weighted 0.89844
wandb:                     +26 ...
wandb: 


wandb: 🚀 View run mdBERTa-25k-2e5-16-256-10 at: https://wandb.ai/nk23041720-kathmandu-university/topic-modeling-25k-experiments/runs/yeiitkhc
wandb: ⭐️ View project at: https://wandb.ai/nk23041720-kathmandu-university/topic-modeling-25k-experiments
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20251017_104641-yeiitkhc/logs
